# PPO Colab ???????
??????????????????????GPU??????????????notebook????????????GPU???
???ZIP??? `MyDrive/cr_rl_ppo/`??????????CPU????

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive', force_remount=True)
BUNDLE = Path('/content/drive/MyDrive/cr_rl_ppo/crrl_ppo_upload_20261116.zip')
ROOT = Path('/content/cr_rl')
assert BUNDLE.is_file(), f'Upload ZIP first: {BUNDLE}'
print('ZIP GB:', round(BUNDLE.stat().st_size / 1e9, 3))


In [ ]:
import hashlib, json, os, shutil, zipfile
from tqdm.auto import tqdm
LOCAL = Path('/content/crrl_ppo_upload.zip')
assert shutil.disk_usage('/content').free > BUNDLE.stat().st_size * 2 + 3 * 1024**3, 'Insufficient local disk space'
# Existing complete files are checked below; an interrupted copy is never promoted.
if not LOCAL.exists() or LOCAL.stat().st_size != BUNDLE.stat().st_size:
    partial = LOCAL.with_suffix('.partial')
    with BUNDLE.open('rb') as source, partial.open('wb') as dest:
        with tqdm(total=BUNDLE.stat().st_size, unit='B', unit_scale=True, desc='Drive to local') as bar:
            for block in iter(lambda: source.read(4 * 1024**2), b''):
                dest.write(block); bar.update(len(block))
    assert partial.stat().st_size == BUNDLE.stat().st_size
    os.replace(partial, LOCAL)
print('Local copy complete; archive and file hashes are checked next.')


In [ ]:
with zipfile.ZipFile(LOCAL) as archive:
    members = archive.infolist()
    for member in members:
        target = (Path('/content') / member.filename).resolve()
        assert target.is_relative_to(ROOT.resolve()), f'Unsafe ZIP member: {member.filename}'
    for member in tqdm(members, desc='Extract + CRC check'):
        archive.extract(member, '/content')
manifest = json.loads((ROOT / 'BUNDLE_MANIFEST.json').read_text())
for row in tqdm(manifest['files'], desc='Verify SHA256'):
    target = (ROOT / row['path']).resolve()
    assert target.is_relative_to(ROOT.resolve())
    assert target.stat().st_size == row['bytes'], row['path']
    digest = hashlib.sha256()
    with target.open('rb') as source:
        for block in iter(lambda: source.read(4 * 1024**2), b''): digest.update(block)
    assert digest.hexdigest() == row['sha256'], f'Checksum mismatch: {row["path"]}'
# Install bundled tables once; training reads local files.
tables = Path.home() / '.crforge/tables/16.635.3'
shutil.copytree(ROOT / 'assets/tables/16.635.3', tables, dirs_exist_ok=True)
print('Verified files:', len(manifest['files']), 'root:', ROOT)


## ????
??Colab??CUDA Torch??cell????????????????Python??????Torch?Java21?????????????????

In [ ]:
import importlib.util, subprocess, sys
assert (3, 12) <= sys.version_info[:2] < (3, 14), sys.version
assert importlib.util.find_spec('torch'), 'Select a Colab runtime with Torch installed'
packages = {'numpy':'numpy>=2', 'jpype':'jpype1>=1.5,<2', 'yaml':'PyYAML>=6',
            'gymnasium':'gymnasium>=1,<2', 'pettingzoo':'pettingzoo>=1.24,<2',
            'msgpack':'msgpack>=1', 'pyarrow':'pyarrow>=18', 'tqdm':'tqdm>=4.66',
            'tensorboard':'tensorboard>=2.14,<3', 'zstandard':'zstandard>=0.23,<1'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(ROOT),
                '-e', str(ROOT / '.cache/vendor/crforge/python')], check=True)
java = subprocess.run(['java','-version'], capture_output=True, text=True) if shutil.which('java') else None
java_text = (java.stderr + java.stdout) if java else ''
if 'version "21' not in java_text:
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','openjdk-21-jdk-headless'], check=True)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.chdir(ROOT)
subprocess.run([sys.executable, 'scripts/train_selfplay.py', '--help'], check=True, stdout=subprocess.DEVNULL)
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Preparation complete. Training is not started; see docs/COLAB_PPO_UPLOAD_20261116.md')
